In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
import numpy as np
import cv2
import os


In [ ]:
DATASET_PATH = r"/content/drive/MyDrive/abns_99/rice_leaf_diseases_dataset"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 25


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


Found 496 images belonging to 6 classes.
Found 123 images belonging to 6 classes.


In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)


Epoch 1/25
31/31 ━━━━━━━━━━━━━━━━━━━━ 212s 6s/step - accuracy: 0.2776 - loss: 1.8083 - val_accuracy: 0.6667 - val_loss: 0.9483
Epoch 2/25
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 525ms/step - accuracy: 0.5995 - loss: 1.0227 - val_accuracy: 0.7561 - val_loss: 0.7652
Epoch 3/25
31/31 ━━━━━━━━━━━━━━━━━━━━ 17s 542ms/step - accuracy: 0.7543 - loss: 0.7579 - val_accuracy: 0.8618 - val_loss: 0.5748
Epoch 4/25
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 524ms/step - accuracy: 0.8255 - loss: 0.5495 - val_accuracy: 0.8374 - val_loss: 0.4871
Epoch 5/25
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 518ms/step - accuracy: 0.8580 - loss: 0.4714 - val_accuracy: 0.8780 - val_loss: 0.4241
Epoch 6/25
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 512ms/step - accuracy: 0.8793 - loss: 0.4202 - val_accuracy: 0.8862 - val_loss: 0.4166
Epoch 7/25
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 511ms/step - accuracy: 0.8887 - loss: 0.3739 - val_accuracy: 0.9106 - val_loss: 0.3244
Epoch 8/25
31/31 ━━━━━━━━━━━━━━━━━━━━ 19s 606ms/step - accuracy: 0.8882 - loss: 0.3433 - val_accura

In [ ]:
model.save("rice_leaf_disease_model.h5")


In [ ]:
CLASS_NAMES = list(train_data.class_indices.keys())

def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)
    class_index = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    return CLASS_NAMES[class_index], confidence


In [ ]:
image_path = r"/content/drive/MyDrive/abns_99/rice_leaf_diseases_dataset/healthy/healthy_val (25).jpg"
disease, confidence = predict_image(image_path)

print("Predicted Disease:", disease)
print("Confidence:", confidence, "%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicted Disease: healthy
Confidence: 94.00704 %


In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, BatchNormalization
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
import numpy as np
import cv2


In [4]:
DATASET_PATH = r"/content/drive/MyDrive/abns_99/rice_leaf_diseases_dataset"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30


In [5]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


Found 496 images belonging to 6 classes.
Found 123 images belonging to 6 classes.


In [6]:
cnn_model = Sequential()

# -------- Block 1 --------
cnn_model.add(Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPooling2D(2,2))

# -------- Block 2 --------
cnn_model.add(Conv2D(64, (3,3), activation='relu'))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPooling2D(2,2))

# -------- Block 3 --------
cnn_model.add(Conv2D(128, (3,3), activation='relu'))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPooling2D(2,2))

# -------- Block 4 --------
cnn_model.add(Conv2D(256, (3,3), activation='relu'))
cnn_model.add(BatchNormalization())
cnn_model.add(MaxPooling2D(2,2))

# -------- Fully Connected --------
cnn_model.add(Flatten())
cnn_model.add(Dense(256, activation='relu'))
cnn_model.add(Dropout(0.5))
cnn_model.add(Dense(train_data.num_classes, activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
cnn_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [8]:
history_cnn = cnn_model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 356s 11s/step - accuracy: 0.4665 - loss: 2.7834 - val_accuracy: 0.1301 - val_loss: 2.0185
Epoch 2/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 509ms/step - accuracy: 0.7222 - loss: 1.5173 - val_accuracy: 0.1301 - val_loss: 3.0959
Epoch 3/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 528ms/step - accuracy: 0.6975 - loss: 1.2162 - val_accuracy: 0.1301 - val_loss: 2.6965
Epoch 4/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 502ms/step - accuracy: 0.8176 - loss: 0.7753 - val_accuracy: 0.1301 - val_loss: 3.0464
Epoch 5/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 505ms/step - accuracy: 0.8239 - loss: 0.6104 - val_accuracy: 0.1382 - val_loss: 3.3149
Epoch 6/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 15s 501ms/step - accuracy: 0.7914 - loss: 0.6009 - val_accuracy: 0.3577 - val_loss: 2.4925
Epoch 7/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 15s 499ms/step - accuracy: 0.8253 - loss: 0.5576 - val_accuracy: 0.4797 - val_loss: 2.7672
Epoch 8/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 15s 496ms/step - accuracy: 0.8841 - loss: 0.4437 - val_accur

In [9]:
cnn_model.save("rice_leaf_disease_cnn_model.h5")


In [10]:
CLASS_NAMES = list(train_data.class_indices.keys())

def predict_with_cnn(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    prediction = cnn_model.predict(img)
    class_index = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    return CLASS_NAMES[class_index], confidence


In [12]:
image_path = r"/content/drive/MyDrive/abns_99/rice_leaf_diseases_dataset/healthy/healthy_val (25).jpg" # Replace with the actual path to your image in Google Drive
disease, confidence = predict_with_cnn(image_path)

print("CNN Prediction:", disease)
print("Confidence:", confidence, "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 933ms/step
CNN Prediction: healthy
Confidence: 98.31784 %
